# Day 11 — Matrix multiplication = a batch of dot products

**Milestone 2:** day 8 scored one query against many docs (matrix–vector). Attention scores **every query against every key** — a whole matrix of dot products, $QK^\top$. That's just matrix multiplication.

## 1. Review (~5 min)

Recall **day 8** (query vs corpus) and **day 9** (top-k).

In [ ]:
import numpy as np
rng = np.random.default_rng(11)

**R1.** (day 8) cosine similarity of a query `q` to every row of `D` → length-`n` scores.

In [ ]:
def r_cosine_sim_to_corpus(q, D):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    n, d = 4, 3
    q = rng.normal(size=d); D = rng.normal(size=(n, d))
    oracle = np.array([np.dot(q, D[i])/(np.linalg.norm(q)*np.linalg.norm(D[i])) for i in range(n)])
    assert np.allclose(np.asarray(fn(q, D)), oracle)
    print("✅ Review R1 passed")

check_r1(r_cosine_sim_to_corpus)

**R2.** (day 9) indices of the `k` largest scores, descending.

In [ ]:
def r_topk(s, k):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    s = rng.normal(size=8)
    assert list(fn(s, 3)) == list(np.argsort(-s)[:3])
    print("✅ Review R2 passed")

check_r2(r_topk)

## 2. Lecture note (~10 min): matrix multiply, demystified

**The itch.** With $n_q$ queries and $n_k$ keys we need *all* $n_q\times n_k$ similarity scores. Looping is slow and clumsy; we want them in one operation.

**The picture.** A matrix product $C = AB$ fills a grid where **each entry is a dot product**:
$$C_{ij} = \sum_k A_{ik} B_{kj} = (\text{row } i \text{ of } A)\cdot(\text{col } j \text{ of } B).$$
Shapes must mesh: $(m, k)\times(k, n) \to (m, n)$ — the inner dimensions cancel.

**Attention's score matrix.** Stack queries as rows of $Q$ (shape $(n_q, d)$) and keys as rows of $K$ (shape $(n_k, d)$). Then
$$ (QK^\top)_{ij} = q_i\cdot k_j, $$
so one product gives **every query–key dot product at once** — a $(n_q, n_k)$ score matrix. (Note the transpose: we need $K^\top$ of shape $(d, n_k)$ so the inner $d$ dimensions match.)

**Knobs & effect.** Matrix multiply *is* batched dot products — the same leap as day 8, one dimension bigger. A vector dotted against itself generalizes to the **Gram matrix** $XX^\top$ of all pairwise dot products (symmetric, with squared norms on its diagonal).

**In the wild.** Every linear layer (`W x`), the $QK^\top$ attention scores, covariance/Gram matrices, and similarity search at scale.

## 3. Exercises (~15 min)

### Exercise 1 — matrix multiply
Return the product `A @ B` for `A` of shape `(m, k)` and `B` of shape `(k, n)`. (Checked against an explicit triple-loop oracle so you see what each entry is.)

In [ ]:
def matmul(A, B):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(4):
        m, k, n = (int(rng.integers(1, 5)) for _ in range(3))
        A = rng.normal(size=(m, k)); B = rng.normal(size=(k, n))
        C = np.asarray(fn(A, B)); assert C.shape == (m, n)
        oracle = np.zeros((m, n))
        for i in range(m):
            for j in range(n):
                oracle[i, j] = sum(A[i, t] * B[t, j] for t in range(k))
        assert np.allclose(C, oracle)
    print("✅ Exercise 1 passed")

check_e1(matmul)

### Exercise 2 — the attention score matrix $QK^\top$
Given `Q` of shape `(n_q, d)` and `K` of shape `(n_k, d)`, return the `(n_q, n_k)` matrix whose `[i, j]` entry is $q_i\cdot k_j$.

In [ ]:
def score_matrix(Q, K):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    nq, nk, d = 3, 4, 5
    Q = rng.normal(size=(nq, d)); K = rng.normal(size=(nk, d))
    S = np.asarray(fn(Q, K)); assert S.shape == (nq, nk)
    for i in range(nq):
        for j in range(nk):
            assert np.allclose(S[i, j], np.dot(Q[i], K[j]))
    print("✅ Exercise 2 passed")

check_e2(score_matrix)

### Exercise 3 — the Gram matrix (a property)
Return $X X^\top$ for `X` of shape `(n, d)`. Confirm it's **symmetric** and that its **diagonal holds each row's squared norm**.

In [ ]:
def gram(X):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    X = rng.normal(size=(5, 3))
    G = np.asarray(fn(X)); assert G.shape == (5, 5)
    assert np.allclose(G, G.T)
    assert np.allclose(np.diag(G), np.sum(X**2, axis=1))
    print("✅ Exercise 3 passed")

check_e3(gram)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_cosine_sim_to_corpus(q, D):
    qn = q / np.linalg.norm(q); Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
    return Dn @ qn

def ref_topk(s, k):
    return np.argsort(-s)[:k]

def ref_matmul(A, B):
    return A @ B

def ref_score_matrix(Q, K):
    return Q @ K.T

def ref_gram(X):
    return X @ X.T

check_r1(ref_cosine_sim_to_corpus)
check_r2(ref_topk)
check_e1(ref_matmul)
check_e2(ref_score_matrix)
check_e3(ref_gram)
print("\nAll reference solutions pass. 🎉  See you on day 12!")